In [55]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("./data/train_cleaned.csv")
print(f"Loaded {len(df):,} rows")

%load_ext autoreload
%autoreload 2

Loaded 891 rows
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
X = df[['Pclass', 'Sex', 'Age', 'Fare']]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [57]:
# Scaling features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Converting to tensors, y must have the shape [N, 1] and [N]
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

In [58]:
# Build the neuron: 4 inputs -> 1 output, then squash to a probability.
# nn.Linear = the weighted sum + bias.  nn.Sigmoid = the squash into 0-1.
# nn.Sequential just runs them in order, one after the other.
model = nn.Sequential(
    nn.Linear(in_features=4, out_features=8),
    # ReLU introduces non-linearity, allowing the network to learn complex rules
    nn.ReLU(),
    nn.Linear(in_features=8, out_features=1),
    nn.Sigmoid(),
)

prob = model(X_train_tensor)
print(prob[:5])

tensor([[0.6140],
        [0.5912],
        [0.5966],
        [0.5878],
        [0.4872]], grad_fn=<SliceBackward0>)


In [59]:
# BCELoss = binary cross-entropy = the log loss you already know. Is used for yes/no classification
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

# One full pass over the data = one "epoch".
for epoch in range(1000):
    optimizer.zero_grad()                  # clear last step's gradients
    output = model(X_train_tensor)         # forward pass: predict on every row
    loss = loss_fn(output, y_train_tensor) # measure how wrong we are
    loss.backward()                        # backprop: work out each weight's blame
    optimizer.step()                       # nudge every weight a little

    if epoch % 10 == 0:
        print(f'Epoch {epoch:4d}, loss {loss.item():.4f}')

Epoch    0, loss 0.7826
Epoch   10, loss 0.4814
Epoch   20, loss 0.4452
Epoch   30, loss 0.4344
Epoch   40, loss 0.4286
Epoch   50, loss 0.4241
Epoch   60, loss 0.4221
Epoch   70, loss 0.4207
Epoch   80, loss 0.4199
Epoch   90, loss 0.4192
Epoch  100, loss 0.4186
Epoch  110, loss 0.4180
Epoch  120, loss 0.4176
Epoch  130, loss 0.4169
Epoch  140, loss 0.4164
Epoch  150, loss 0.4160
Epoch  160, loss 0.4156
Epoch  170, loss 0.4151
Epoch  180, loss 0.4147
Epoch  190, loss 0.4139
Epoch  200, loss 0.4129
Epoch  210, loss 0.4114
Epoch  220, loss 0.4099
Epoch  230, loss 0.4087
Epoch  240, loss 0.4077
Epoch  250, loss 0.4068
Epoch  260, loss 0.4061
Epoch  270, loss 0.4053
Epoch  280, loss 0.4045
Epoch  290, loss 0.4036
Epoch  300, loss 0.4030
Epoch  310, loss 0.4023
Epoch  320, loss 0.4021
Epoch  330, loss 0.4023
Epoch  340, loss 0.4015
Epoch  350, loss 0.4014
Epoch  360, loss 0.4011
Epoch  370, loss 0.4010
Epoch  380, loss 0.4018
Epoch  390, loss 0.4008
Epoch  400, loss 0.4007
Epoch  410, loss

In [60]:
with torch.no_grad():
    pred = model(X_test_tensor)
    rounded_pred = (pred > 0.5).float()

    matches = rounded_pred == y_test_tensor

    correct_count = matches.sum().item()
    total_count = len(y_test_tensor)

    acc = correct_count / total_count
    print(f"Accuracy: {acc * 100:.2f}%")


Accuracy: 80.45%
